In [ ]:
!pip install -q kailash kailash-ml polars torch torchvision matplotlib python-dotenv scikit-learn plotly


# MLFP05 — Exercise 7, Part 1: Training a CNN from Scratch (Baseline)


**WHAT YOU'LL LEARN:**
After completing this section, you will be able to:
- Build and train a CNN from random initialisation on CIFAR-10
- Understand why training from scratch requires massive labelled data
- Visualise learned filters at different layers (random early vs
structured later) and feature representations with t-SNE
- Quantify the data bottleneck that motivates transfer learning


**PREREQUISITES: M5/ex_2 (CNNs and PyTorch), M5/ex_1 (ExperimentTracker).**

**ESTIMATED TIME: ~25 min**



In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn

from shared.mlfp05.ex_7 import (
    BATCH_SIZE,
    CLASS_NAMES,
    EPOCHS,
    INPUT_SIZE,
    N_CLASSES,
    OUTPUT_DIR,
    count_params,
    create_visualizer,
    device,
    extract_features,
    compute_tsne,
    cluster_quality,
    init_engines,
    load_cifar10,
    plot_tsne,
    save_training_plots,
    train_model,
)


## THEORY — Why Training from Scratch Is Hard
A CNN initialised with random weights knows nothing about the visual
world. Every filter — edge detectors, texture recognisers, shape
combinations — must be learned entirely from your training data.
The ImageNet revolution (2012) showed that deep CNNs can learn these
filters, but they needed 1.28 MILLION labelled images to do it.
Most real-world projects have 1,000 - 50,000 labelled images, not
millions. This creates the "data bottleneck": the model has enough
capacity to learn, but not enough examples to learn FROM.
This baseline quantifies the problem. We train a small CNN from
scratch on CIFAR-10 (50K images) and measure its accuracy. In the
next section, we'll compare this against a ResNet-18 that starts
with ImageNet knowledge — and the gap will motivate everything that
follows.


In [ ]:
print("\n" + "=" * 70)
print("  PART 1: Training a CNN from Scratch (Baseline)")
print("=" * 70)


## TASK 1 — Load CIFAR-10 and initialise kailash-ml engines


In [ ]:
train_set, val_set, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()


### Checkpoint 1 ─────────────────────────────────────────────────────


In [ ]:
assert len(train_set) == 50000, (
    f"Expected full 50K CIFAR-10, got {len(train_set)}. "
    "Transfer learning exercises need the full dataset."
)
assert len(val_set) == 10000, "CIFAR-10 test set should be 10K"
assert tracker is not None, "ExperimentTracker should be initialised"
# INTERPRETATION: We use the full 50K training set so that the
# from-scratch baseline has every possible advantage. Even with all
# 50K images, the scratch CNN will be limited because it must learn
# every visual feature from these images alone.
print("\n--- Checkpoint 1 passed --- CIFAR-10 loaded, engines ready\n")


## TASK 2 — Build a from-scratch CNN
A small CNN trained from random initialisation. We keep the
architecture simple (3 conv layers + classifier) to isolate the
effect of pre-training vs architecture advantage.


In [ ]:
def build_scratch_cnn(n_classes: int = N_CLASSES) -> nn.Module:
    """Baseline: a small CNN trained from random init."""
    # TODO: Build a nn.Sequential CNN with:
    #   - Conv2d(3, 32, 3, padding=1) + BatchNorm2d(32) + ReLU + MaxPool2d(2)
    #   - Conv2d(32, 64, 3, padding=1) + BatchNorm2d(64) + ReLU + MaxPool2d(2)
    #   - Conv2d(64, 128, 3, padding=1) + BatchNorm2d(128) + ReLU + AdaptiveAvgPool2d(1)
    #   - Flatten + Dropout(0.3) + Linear(128, n_classes)
    # Hint: nn.Sequential takes all layers as positional arguments
    pass  # Replace with your implementation


scratch_model = build_scratch_cnn()
n_total = count_params(scratch_model)
print(f"  From-scratch CNN: {n_total:,} total parameters (all trainable)")


### Checkpoint 2 ─────────────────────────────────────────────────────


In [ ]:
assert n_total > 50000, f"CNN should have >50K params, got {n_total:,}"
assert n_total < 500000, f"CNN should be small (<500K params), got {n_total:,}"
# INTERPRETATION: This is a deliberately small CNN. We keep it small
# so the comparison with transfer learning is about knowledge, not
# architecture size. The scratch CNN has to learn edge detectors,
# texture recognisers, and shape combinations all from 50K images.
print("--- Checkpoint 2 passed --- CNN architecture built\n")


## TASK 3 — Train the from-scratch CNN with ExperimentTracker


In [ ]:
print("\n" + "=" * 70)
print("  TRAINING: From-Scratch CNN on full CIFAR-10 (50K)")
print("=" * 70)

scratch_losses, scratch_accs, scratch_train_accs = train_model(
    scratch_model,
    "cnn_from_scratch",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)

best_scratch = max(scratch_accs)


### Checkpoint 3 ─────────────────────────────────────────────────────


In [ ]:
assert len(scratch_losses) == EPOCHS, "Should have per-epoch losses"
assert (
    best_scratch > 0.30
), f"Scratch val accuracy {best_scratch:.3f} below 0.30 -- check training"
# INTERPRETATION: With 50K images and 8 epochs, the scratch CNN
# reaches a reasonable but not stellar accuracy. Every percentage point
# required the model to discover visual features from raw pixels.
# In part 2, we'll see how much ImageNet pre-training changes this.
print(f"\n  From-scratch best val_acc: {best_scratch:.3f}")
print("--- Checkpoint 3 passed --- baseline training complete\n")


## TASK 4 — Visualise: Learned filters and t-SNE feature space
Two visualisations reveal what the scratch CNN actually learned:
1. LEARNED FILTERS — The first conv layer's 32 filters. In a
randomly-initialised network, these look like noise. After
training, some develop into crude edge/colour detectors, but
they're much noisier than pre-trained ResNet filters.
2. t-SNE FEATURE SPACE — We extract the 128-dimensional features
from the penultimate layer and project to 2D. If the CNN learned
good representations, classes should cluster. Scratch-trained
features typically show more overlap between classes.


In [ ]:
# -- Learned filters visualisation --
print("-- Visualising learned convolutional filters --")

# TODO: Extract first conv layer weights and visualise them
# Steps:
#   1. Get weights: scratch_model[0].weight.data.cpu()  -> shape (32, 3, 3, 3)
#   2. Pick first 16 filters (n_filters = min(16, ...))
#   3. For each filter: normalise to [0,1], average across RGB channels
#   4. Create a go.Figure() with go.Heatmap traces for each filter
#   5. Save to OUTPUT_DIR / "01_scratch_filters.html"
# Hint: filt_norm = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
# Hint: filt_gray = filt_norm.mean(dim=0).numpy()

conv1_weights = scratch_model[0].weight.data.cpu()  # shape: (32, 3, 3, 3)
n_filters = min(16, conv1_weights.shape[0])

fig_filters = go.Figure()
# TODO: Loop through n_filters, normalise each filter, add Heatmap traces
# Hint: Use np.flipud() for display, colorscale="Greys"
pass  # Replace with your filter visualisation loop

fig_filters.update_layout(
    title="From-Scratch CNN: Learned First-Layer Filters (noisy, unstructured)",
    template="plotly_white",
    width=600,
    height=600,
)
filters_path = OUTPUT_DIR / "01_scratch_filters.html"
fig_filters.write_html(str(filters_path))
print(f"  Saved: {filters_path}")

# -- t-SNE feature space --
print("-- Extracting features for t-SNE visualisation --")
scratch_feats, scratch_labels = extract_features(scratch_model, val_loader)
print(f"  Scratch features shape: {scratch_feats.shape}")

coords_scratch = compute_tsne(scratch_feats)
cq_scratch = cluster_quality(coords_scratch, scratch_labels)

print(f"  t-SNE cluster quality (lower = better): {cq_scratch:.4f}")
print(f"\n  Class centroids (first 5 classes):")
for c in range(min(5, N_CLASSES)):
    mask = scratch_labels == c
    centroid = coords_scratch[mask].mean(axis=0)
    print(f"    {CLASS_NAMES[c]:>12}: ({centroid[0]:+.1f}, {centroid[1]:+.1f})")

tsne_path = OUTPUT_DIR / "01_scratch_tsne.html"
plot_tsne(coords_scratch, scratch_labels, "t-SNE: From-Scratch CNN Features", tsne_path)

# Save training curves
viz = create_visualizer()
save_training_plots(
    viz,
    {"scratch loss": scratch_losses, "scratch val_acc": scratch_accs},
    OUTPUT_DIR / "01_scratch_training.html",
)


### Checkpoint 4 ─────────────────────────────────────────────────────


In [ ]:
assert scratch_feats.shape[0] > 0, "Should have extracted features"
assert coords_scratch.shape == (scratch_feats.shape[0], 2), "t-SNE should produce 2D"
# INTERPRETATION: The t-SNE plot likely shows overlapping clusters —
# classes that the scratch CNN can't fully separate. This is what
# "learning from scratch" looks like: with only 50K images and 8
# epochs of training, the feature representations are noisy and
# incomplete. Compare this to the transfer learning t-SNE in part 2.
print("\n--- Checkpoint 4 passed --- visualisations complete\n")


## TASK 5 — Apply: Small Singapore Startup with Limited Data
SCENARIO: You're the ML engineer at a Singapore startup (like Carro
or ShopBack) trying to build an image classifier for product photos.
You only have 5,000 labelled images — 10% of CIFAR-10.
Question: What happens to from-scratch accuracy with limited data?


In [ ]:
print("\n" + "=" * 70)
print("  APPLY: Singapore Startup with 5,000 Labelled Images")
print("=" * 70)

# TODO: Simulate the startup scenario by training on only 10% of CIFAR-10
# Steps:
#   1. Create a random subset of 5,000 indices from train_set
#   2. Build a Subset and DataLoader from those indices
#   3. Train a fresh build_scratch_cnn() on the limited data
#   4. Compare best accuracy with the full-data result
# Hint: rng = np.random.default_rng(42)
# Hint: indices = rng.choice(len(train_set), size=5000, replace=False).tolist()
# Hint: Use Subset(train_set, indices) and DataLoader

rng = np.random.default_rng(42)
n_startup = 5000
# TODO: Create indices, subset, and loader
# TODO: Build and train a fresh scratch CNN on the limited data
# TODO: Store the best accuracy in best_startup
from torch.utils.data import Subset, DataLoader as DL

indices = (
    None  # TODO: rng.choice(len(train_set), size=n_startup, replace=False).tolist()
)
startup_subset = None  # TODO: Subset(train_set, indices)
startup_loader = (
    None  # TODO: DL(startup_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
)

startup_model = build_scratch_cnn()
# TODO: Train startup_model using train_model() with startup_loader
# startup_losses, startup_accs, _ = train_model(...)
startup_losses, startup_accs = [], []  # Replace with actual training
best_startup = 0.0  # TODO: max(startup_accs) after training

print(f"\n  === Startup Scenario Results ===")
print(f"  Full data (50K):    {best_scratch:.1%} accuracy")
print(f"  Startup (5K):       {best_startup:.1%} accuracy")
print(f"  Accuracy drop:      {best_scratch - best_startup:+.1%}")
print()
print(f"  BUSINESS IMPACT:")
print(f"  With only 5,000 labelled images, from-scratch training loses")
print(f"  significant accuracy. For a product classifier at a Singapore")
print(f"  startup, this means:")
print(f"    - More misclassified products shown to customers")
print(f"    - Higher rate of manual review needed")
print(f"    - Labelling 50K images costs ~S$25,000-50,000 (S$0.50-1.00/label)")
print(f"  This is exactly the problem transfer learning solves (Part 2).")


### Checkpoint 5 ─────────────────────────────────────────────────────


In [ ]:
assert (
    best_startup > 0.15
), f"Even with 5K data, should beat random chance (acc={best_startup:.3f})"
assert (
    best_startup < best_scratch
), "With less data, accuracy should be lower than full-data training"
# INTERPRETATION: The accuracy drop from 50K to 5K images quantifies
# the data bottleneck. This is the core motivation for transfer
# learning: instead of collecting 10x more labelled data (expensive),
# we can reuse features already learned from ImageNet (free).
print("\n--- Checkpoint 5 passed --- startup scenario complete\n")


## CLEANUP


In [ ]:
import asyncio

await conn.close()


## REFLECTION


In [ ]:
print("\n" + "=" * 70)
print("  PART 1 COMPLETE — What You've Learned")
print("=" * 70)
print(
    f"""
  [x] Built and trained a from-scratch CNN on CIFAR-10 ({n_total:,} params)
  [x] Achieved {best_scratch:.1%} val accuracy with full 50K training data
  [x] Visualised learned filters (noisy, unstructured after 8 epochs)
  [x] Mapped feature space with t-SNE (cluster quality: {cq_scratch:.3f})
  [x] Quantified the startup data bottleneck: {best_startup:.1%} with only 5K images

  KEY INSIGHT: Training from scratch requires massive labelled datasets.
  A Singapore startup with 5,000 images loses significant accuracy because
  the CNN must discover every visual feature from limited examples.

  NEXT: Part 2 uses a pre-trained ResNet-18 backbone to solve this exact
  problem. The pre-trained model brings ImageNet knowledge (edges, textures,
  shapes) and only needs to learn the final classification layer.
"""
)
